# EHR Mamba Embedding on MIMIC-IV

This notebook trains the EHR Mamba model with the full paper §2.2 embedding on
MIMIC-IV in-hospital mortality prediction.

The pipeline uses three project files:
- `mortality_prediction_ehrmamba_mimic4.py` — `MIMIC4EHRMambaMortalityTask`, `LabQuantizer`, `collate_ehr_mamba_batch`
- `ehrmamba_embedding.py` — `EHRMambaEmbedding` (7-component §2.2 / Eq.§1 embedding)
- `ehrmamba_vi.py` — `EHRMamba` model

**Sections 1–7:** MIMIC-IV dev subset (`dev=True`) — fast iteration and debugging.

**Ablation Study (Sections 8–14):** The second half of this notebook repeats the
identical pipeline on the publicly available Synthetic MIMIC-III demo dataset
to verify that the same task, embedding, and model code generalises beyond
MIMIC-IV. Two compatibility shims are applied inside a `MIMIC3DatasetICD`
subclass—no changes to the task, embedding, or model files are required:

1. **Procedure codes** — MIMIC-III stores ICD procedure codes under `icd9_code`;
   the task expects `icd_code`. A `preprocess_procedures_icd` hook renames the
   column at load time.
2. **Patient age** — MIMIC-III has no `anchor_age` / `anchor_year` fields (only
   `dob`). The MIMIC-III demo dates are shifted to 2102–2202, so a synthetic
   `anchor_year = 2160` (mid-dataset reference) and
   `anchor_age = 2160 − birth_year` are derived in `preprocess_patients`.
   This choice retains 97 / 100 demo patients under `min_age = 18`
   (vs. 63 / 100 with `anchor_year = 2102`).

**Full MIMIC-IV (Sections 15–21):** Same pipeline as Sections 1–7 against the full
MIMIC-IV dataset at `../../data/mimic4` (`dev=True`). Variable names are prefixed
with `full_` to avoid shadowing earlier objects.


In [1]:
from pyhealth.datasets import MIMIC4EHRDataset, split_by_sample
from pyhealth.trainer import Trainer
from pyhealth.tasks.mortality_prediction_ehrmamba_mimic4 import (
    MIMIC4EHRMambaMortalityTask,
    LabQuantizer,
    collate_ehr_mamba_batch,
)
from pyhealth.models.ehrmamba_vi import EHRMamba
from torch.utils.data import DataLoader
import torch


C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\trainer.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


## 1. Load MIMIC-IV Dataset

Set `DEV_MODE = False` to use the full dataset.


In [ ]:
DATA_ROOT = '../../data/mimic4_demo'
CACHE_ROOT = '../../data/mimic4_demo/cache'

dataset = MIMIC4EHRDataset(
    root=DATA_ROOT,
    cache_dir=CACHE_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    dev=True,
)


Using default EHR config: C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\datasets\configs\mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 563.2 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from ../../data/mimic4 (dev mode: True)
Using provided cache_dir: ..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3
Memory usage After initializing mimic4_ehr: 563.4 MB


## 2. Fit Lab Quantizer and Set Task

`LabQuantizer` bins continuous lab values into 5 quantile tokens per `itemid`
(paper Appendix B). `MIMIC4EHRMambaMortalityTask` builds the §2.1 token
sequence: `[CLS] [VS] events [VE] [REG] [W?] ...` with in-hospital mortality
as the label.


In [3]:
# Fit LabQuantizer on all patients (fit only on train split to avoid leakage in production)
lab_quantizer = LabQuantizer(n_bins=5)
lab_quantizer.fit_from_patients(list(dataset.iter_patients()))


task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=lab_quantizer
)
sample_dataset = dataset.set_task(task)
train_dataset, val_dataset, test_dataset = split_by_sample(
    sample_dataset, ratios=[0.5, 0.2, 0.3]
)


Found cached event dataframe: ..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3\global_event_df.parquet
Setting task MIMIC4EHRMambaMortality for mimic4_ehr base dataset...
Task cache paths: task_df=..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3\tasks\MIMIC4EHRMambaMortality_92fc5b89-c3d8-57dd-aec0-2dcc50a55931\task_df.ld, samples=..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3\tasks\MIMIC4EHRMambaMortality_92fc5b89-c3d8-57dd-aec0-2dcc50a55931\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 1000 patients. (Polars threads: 12)


  0%|          | 0/1000 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 1000/1000 [00:06<00:00, 158.93it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


Label label vocab: {0: 0, 1: 1}
Processing samples and saving to ..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3\tasks\MIMIC4EHRMambaMortality_92fc5b89-c3d8-57dd-aec0-2dcc50a55931\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 620 samples. (0 to 620)


  0%|          | 0/620 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1', 'no_header_tensor:1', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 620/620 [00:00<00:00, 834.55it/s]

Worker 0 finished processing samples.
Cached processed samples to ..\..\data\mimic4\cache\d2d872b8-e4ee-5711-9861-11e729358fd3\tasks\MIMIC4EHRMambaMortality_92fc5b89-c3d8-57dd-aec0-2dcc50a55931\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


## 3. Create Data Loaders

`collate_ehr_mamba_batch` right-pads variable-length sequences and stacks the six
EHR Mamba tensor fields (`input_ids`, `token_type_ids`, `time_stamps`, `ages`,
`visit_orders`, `visit_segments`) into `(B, L_max)` tensors.


In [4]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)


## 4. Initialize EHRMamba Model

`use_ehr_mamba_embedding=True` replaces PyHealth's default embedding with
`EHRMambaEmbeddingAdapter(EHRMambaEmbedding(...))` — the full 7-component
Equation 1 embedding from §2.2.


In [5]:
model = EHRMamba(
    dataset=sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
trainer = Trainer(model=model, metrics=['roc_auc', 'pr_auc'])
print(model)


EHRMamba(
  (embedding_model): EHRMambaEmbeddingAdapter(
    (embedding): EHRMambaEmbedding(
      (word_embeddings): Embedding(3651, 128, padding_idx=0)
      (token_type_embeddings): Embedding(10, 128)
      (visit_order_embeddings): Embedding(512, 128)
      (visit_segment_embeddings): VisitEmbedding(
        (embedding): Embedding(3, 128)
      )
      (position_embeddings): Embedding(4096, 128)
      (time_embeddings): TimeEmbeddingLayer()
      (age_embeddings): TimeEmbeddingLayer()
      (scale_back_concat_layer): Linear(in_features=192, out_features=128, bias=True)
      (tanh): Tanh()
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (blocks): ModuleDict(
    (input_ids): ModuleList(
      (0-1): 2 x MambaBlock(
        (norm): RMSNorm()
        (in_proj): Linear(in_features=128, out_features=512, bias=True)
        (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,))
        (

## 5. Train

In [6]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=50,
    monitor='roc_auc',
    optimizer_params={'lr': 1e-4},
)


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x0000018FB4999D60>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/10 [00:00<?, ?it/s]

--- Train epoch-0, step-10 ---
loss: 0.5392


Evaluation: 100%|██████████| 4/4 [00:40<00:00, 10.12s/it]

--- Eval epoch-0, step-10 ---
roc_auc: 0.3798
pr_auc: 0.0374
loss: 0.4016
New best roc_auc score (0.3798) at epoch-0, step-10



Epoch 1 / 50:   0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6. Evaluate on Test Set

In [7]:
trainer.evaluate(test_loader)


Evaluation: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:00<00:00,  4.56it/s]
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


{'roc_auc': nan, 'pr_auc': 0.0, 'loss': 0.6569212675094604}

## 7. Get Patient Embeddings

Pass `embed=True` to retrieve pooled patient representations from the last
Mamba block. These can be used for downstream tasks or visualisation.


In [8]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch, embed=True)
    embeddings = output['embed']
    print(f'Patient embeddings shape: {embeddings.shape}')


Patient embeddings shape: torch.Size([5, 128])


In [9]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    output = model(**batch)
    print(output['y_prob'])   # predicted mortality probability per patient
    print(output['y_true'])   # ground truth labels

tensor([[0.3944],
        [0.4577],
        [0.4686],
        [0.5737],
        [0.4966]])
tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])


---

## Ablation Study: MIMIC-III Demo Dataset

The cells below repeat **the exact same task, embedding, and model** used in
Sections 1–7 above, but load data from the publicly available Synthetic
MIMIC-III demo hosted by PyHealth. This ablation confirms that the EHRMamba
pipeline is not tightly coupled to MIMIC-IV and can run on MIMIC-III with
no changes to the task, embedding, or model code.

**Compatibility shims applied in `MIMIC3DatasetICD`:**

| Issue | Root cause | Fix |
|---|---|---|
| Missing `icd_code` column | MIMIC-III uses `icd9_code` | `preprocess_procedures_icd` hook renames column at load time |
| Missing `anchor_age` / `anchor_year` | MIMIC-III uses `dob` only | `preprocess_patients` derives `anchor_year=2160`, `anchor_age=2160−birth_year` |

**Why `anchor_year = 2160`:** The MIMIC-III demo dates are shifted to the future
(DOBs 1844–2181, admissions 2102–2202, median birth year ≈2073). Using 2102
(earliest admission) as the anchor year left 37 % of patients below `min_age = 18`
because most patients were not yet born by then. Choosing 2160 (mid-dataset
reference, median `anchor_age` ≈87) retains 97 / 100 demo patients.

**Residual known limitation:** Patients aged >89 have their `dob` shifted to
∼1844 by MIMIC-III de-identification, producing `anchor_age` ≈316. These
patients pass `min_age` but carry nonsensical age embeddings—a MIMIC-III data
artifact that cannot be corrected without the original records.


### 8. Load MIMIC-III Demo Dataset

Uses the synthetic MIMIC-III dataset served by PyHealth — no local download
required. `dev=True` limits the dataset to 1 000 patients.

In [2]:
from pyhealth.datasets import MIMIC3Dataset
import narwhals as nw

DATA_ROOT = '../../data/mimic3_demo/datasets'
CACHE_ROOT = '../../data/mimic3_demo/cache'


class MIMIC3DatasetICD(MIMIC3Dataset):
    """MIMIC3Dataset adapted for MIMIC4EHRMambaMortalityTask.

    Applies two compatibility shims so that MIMIC-III data can be fed into
    the MIMIC-IV task pipeline without modifying any task, embedding, or
    model files.

    Shim 1 - Procedure codes:
        MIMIC-III stores ICD procedure codes under the column ``icd9_code``;
        the task expects ``icd_code``. A ``preprocess_procedures_icd`` hook
        renames the column, and ``__init__`` patches the config attribute
        list so the renamed column survives attribute selection.

    Shim 2 - Patient age (anchor semantics):
        MIMIC-IV exposes ``anchor_age`` and ``anchor_year`` directly.
        MIMIC-III only has ``dob``. The MIMIC-III demo dates are shifted to
        the future (DOBs 1844-2181, admissions 2102-2202). A synthetic
        ``anchor_year = 2160`` is chosen as a mid-dataset reference because
        it retains 97 / 100 demo patients under ``min_age = 18`` (vs. only
        63 / 100 with ``anchor_year = 2102``). ``anchor_age`` is then
        derived as ``2160 - birth_year``, satisfying the per-visit age
        identity used by the task::

            age_at_visit = anchor_age + (admit_year - anchor_year)
                         = (2160 - birth_year) + (admit_year - 2160)
                         = admit_year - birth_year

    Note:
        Patients aged > 89 at admission have their ``dob`` shifted to
        ~1844 by MIMIC-III de-identification, yielding ``anchor_age``
        ~316. These pass ``min_age`` but carry nonsensical age
        embeddings -- a MIMIC-III data artifact.

        ``nw.lit()`` has inconsistent support on the Dask backend, so
        ``preprocess_patients`` operates on the native Dask DataFrame
        via ``df.to_native()`` rather than narwhals expressions.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Rename icd9_code -> icd_code in the procedures_icd attribute list.
        proc_attrs = self.config.tables["procedures_icd"].attributes
        if "icd9_code" in proc_attrs:
            proc_attrs[proc_attrs.index("icd9_code")] = "icd_code"
        # Expose anchor_year and anchor_age so they survive attribute selection.
        pat_attrs = self.config.tables["patients"].attributes
        for col in ("anchor_year", "anchor_age"):
            if col not in pat_attrs:
                pat_attrs.append(col)

    def preprocess_procedures_icd(self, df: nw.LazyFrame) -> nw.LazyFrame:
        """Rename icd9_code to icd_code so procedure tokens are populated.

        Args:
            df: Raw ``procedures_icd`` table as a narwhals LazyFrame.

        Returns:
            LazyFrame with ``icd9_code`` renamed to ``icd_code``.
        """
        return df.rename({"icd9_code": "icd_code"})

    def preprocess_patients(self, df: nw.LazyFrame) -> nw.LazyFrame:
        """Derive anchor_year and anchor_age from dob for MIMIC-III records.

        Uses the native Dask DataFrame directly because ``nw.lit()`` is
        unreliable on the Dask backend.

        Args:
            df: Raw ``patients`` table as a narwhals LazyFrame.

        Returns:
            LazyFrame with two additional columns: ``anchor_year``
            (constant 2160) and ``anchor_age`` (``2160 - birth_year``).
        """
        native = df.to_native()
        birth_year = native["dob"].astype(str).str[:4].astype(int)
        native = native.assign(
            anchor_year=2160,
            anchor_age=(2160 - birth_year),
        )
        return nw.from_native(native)


mimic3_dataset = MIMIC3DatasetICD(
    root=DATA_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    cache_dir=CACHE_ROOT,
    dev=True,
)


No config path provided, using default config
Duplicate table names in tables list. Removing duplicates.
Initializing mimic3 dataset from ../../data/mimic3_demo/datasets (dev mode: True)
Using provided cache_dir: ..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f


C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\PyHealth\pyhealth\datasets\mimic3.py:59: UserWarning: Events from prescriptions table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


### 9. Fit Lab Quantizer and Set Task

Same `LabQuantizer` and `MIMIC4EHRMambaMortalityTask` as Section 2.
The task reads `hadm_id` (present in both MIMIC-III and MIMIC-IV) to
filter events per admission.

In [3]:
abl_lab_quantizer = LabQuantizer(n_bins=5)
abl_lab_quantizer.fit_from_patients(list(mimic3_dataset.iter_patients()))

# anchor_year=2160 retains 97/100 demo patients at min_age=18
# (vs. 63/100 with anchor_year=2102). See MIMIC3DatasetICD for details.
abl_task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=abl_lab_quantizer,
)
abl_sample_dataset = mimic3_dataset.set_task(abl_task)
abl_train_ds, abl_val_ds, abl_test_ds = split_by_sample(
    abl_sample_dataset, ratios=[0.5, 0.2, 0.3]
)


No cached event dataframe found. Creating: ..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f\global_event_df.parquet
Scanning table: patients from C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\PATIENTS.csv.gz
Preprocessing table: patients with preprocess_patients
Scanning table: admissions from C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\ADMISSIONS.csv.gz
Scanning table: prescriptions from C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\PRESCRIPTIONS.csv.gz
Joining with table: C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\ADMISSIONS.csv.gz
Scanning table: procedures_icd from C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\PROCEDURES_ICD.csv.gz
Preprocessing table: procedures_icd with preprocess_procedures_icd
Joining with table: C:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\data\mimic3_demo\datasets\ADMISSIONS.csv.gz
Scanning table:

c:\Users\aravn\OneDrive\Desktop\EHR_DLH_PROJECT\.venv\Lib\site-packages\dask\dataframe\core.py:382: UserWarning: Insufficient elements for `head`. 1000 elements requested, only 15 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


Setting task MIMIC4EHRMambaMortality for mimic3 base dataset...
Task cache paths: task_df=..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f\tasks\MIMIC4EHRMambaMortality_1221ed0a-3907-59b3-b835-ce5cb36a7803\task_df.ld, samples=..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f\tasks\MIMIC4EHRMambaMortality_1221ed0a-3907-59b3-b835-ce5cb36a7803\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 15 patients. (Polars threads: 12)


  0%|          | 0/15 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


...

Label label vocab: {0: 0, 1: 1}
Processing samples and saving to ..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f\tasks\MIMIC4EHRMambaMortality_1221ed0a-3907-59b3-b835-ce5cb36a7803\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 14 samples. (0 to 14)


  0%|          | 0/14 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1', 'no_header_tensor:1', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 14/14 [00:00<00:00, 568.11it/s]

Worker 0 finished processing samples.
Cached processed samples to ..\..\data\mimic3_demo\cache\700267c1-731c-5cce-8bd6-614bfddfd37f\tasks\MIMIC4EHRMambaMortality_1221ed0a-3907-59b3-b835-ce5cb36a7803\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


### 10. Create Data Loaders

Identical to Section 3 — `collate_ehr_mamba_batch` handles variable-length
sequences from MIMIC-III just as it does for MIMIC-IV.

In [4]:
abl_train_loader = DataLoader(
    abl_train_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
abl_val_loader = DataLoader(
    abl_val_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
abl_test_loader = DataLoader(
    abl_test_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)

### 11. Initialize EHRMamba Model

Same architecture as Section 4 — `use_ehr_mamba_embedding=True` activates
the full 7-component §2.2 embedding. The vocabulary is rebuilt from the
MIMIC-III sample dataset, so `word_embeddings` will have a different
vocab size than the MIMIC-IV model.

In [5]:
abl_model = EHRMamba(
    dataset=abl_sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
abl_trainer = Trainer(model=abl_model, metrics=["roc_auc", "pr_auc"])
print(abl_model)

EHRMamba(
  (embedding_model): EHRMambaEmbeddingAdapter(
    (embedding): EHRMambaEmbedding(
      (word_embeddings): Embedding(254, 128, padding_idx=0)
      (token_type_embeddings): Embedding(10, 128)
      (visit_order_embeddings): Embedding(512, 128)
      (visit_segment_embeddings): VisitEmbedding(
        (embedding): Embedding(3, 128)
      )
      (position_embeddings): Embedding(4096, 128)
      (time_embeddings): TimeEmbeddingLayer()
      (age_embeddings): TimeEmbeddingLayer()
      (scale_back_concat_layer): Linear(in_features=192, out_features=128, bias=True)
      (tanh): Tanh()
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (blocks): ModuleDict(
    (input_ids): ModuleList(
      (0-1): 2 x MambaBlock(
        (norm): RMSNorm()
        (in_proj): Linear(in_features=128, out_features=512, bias=True)
        (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,))
        (o

### 12. Train

In [6]:
abl_trainer.train(
    train_dataloader=abl_train_loader,
    val_dataloader=abl_val_loader,
    epochs=50,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-4},
)

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x00000155CC0B6CC0>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-0, step-1 ---
loss: 0.7755


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 32.08it/s]

--- Eval epoch-0, step-1 ---
roc_auc: 1.0000
pr_auc: 1.0000
loss: 0.6162
New best roc_auc score (1.0000) at epoch-0, step-1



...

--- Eval epoch-48, step-49 ---
roc_auc: 1.0000
pr_auc: 1.0000
loss: 0.9562



Epoch 49 / 50:   0%|          | 0/1 [00:00<?, ?it/s]

--- Train epoch-49, step-50 ---
loss: 0.0161


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 36.17it/s]

--- Eval epoch-49, step-50 ---
roc_auc: 1.0000
pr_auc: 1.0000
loss: 0.9839
Loaded best model


### 13. Evaluate on Test Set

In [7]:
abl_trainer.evaluate(abl_test_loader)


Evaluation: 100%|██████████| 1/1 [00:00<00:00, 30.15it/s]


{'roc_auc': 0.33333333333333337, 'pr_auc': 0.45, 'loss': 0.7039042711257935}

### 14. Get Patient Embeddings

Pooled patient representations from the MIMIC-III model — same API as
Section 7. Embedding shape will be `(B, 128)` regardless of dataset.

In [8]:
abl_model.eval()
with torch.no_grad():
    abl_batch = next(iter(abl_test_loader))
    abl_output = abl_model(**abl_batch, embed=True)
    print(f"Patient embeddings shape: {abl_output['embed'].shape}")
    print(f"Predicted probabilities:  {abl_output['y_prob'].squeeze()}")
    print(f"Ground truth labels:      {abl_output['y_true'].squeeze()}")

Patient embeddings shape: torch.Size([5, 128])
Predicted probabilities:  tensor([0.4690, 0.4712, 0.4382, 0.3121, 0.3189])
Ground truth labels:      tensor([1., 0., 0., 1., 0.])


---

## Full MIMIC-IV Dataset

The cells below repeat **the exact same pipeline** as Sections 1–7 above,
but against the full MIMIC-IV dataset at `../../data/mimic4`. Variable
names are prefixed with `full_` to avoid shadowing the dev-mode objects
from earlier sections.

Training the full dataset requires significantly more memory and time than
the 1 000-patient dev subset. A GPU is strongly recommended.


### 15. Load Full MIMIC-IV Dataset


In [ ]:
FULL_DATA_ROOT = '../../data/mimic4'
FULL_CACHE_ROOT = '../../data/mimic4/cache'

full_dataset = MIMIC4EHRDataset(
    root=FULL_DATA_ROOT,
    cache_dir=FULL_CACHE_ROOT,
    tables=['patients', 'admissions', 'procedures_icd', 'prescriptions', 'labevents'],
    dev=True,
)


### 16. Fit Lab Quantizer and Set Task

Same `LabQuantizer` and `MIMIC4EHRMambaMortalityTask` as Section 2.


In [ ]:
# Fit LabQuantizer on the full dataset. In production, fit on the train
# split only to avoid label leakage into validation / test sets.
full_lab_quantizer = LabQuantizer(n_bins=5)
full_lab_quantizer.fit_from_patients(list(full_dataset.iter_patients()))

full_task = MIMIC4EHRMambaMortalityTask(
    min_age=18,
    lab_quantizer=full_lab_quantizer,
)
full_sample_dataset = full_dataset.set_task(full_task)
full_train_ds, full_val_ds, full_test_ds = split_by_sample(
    full_sample_dataset, ratios=[0.7, 0.1, 0.2]
)


### 17. Create Data Loaders

Identical to Section 3. Increase `batch_size` if GPU memory allows.


In [ ]:
full_train_loader = DataLoader(
    full_train_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
full_val_loader = DataLoader(
    full_val_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)
full_test_loader = DataLoader(
    full_test_ds, batch_size=32, shuffle=False,
    collate_fn=collate_ehr_mamba_batch,
)


### 18. Initialize EHRMamba Model

Same architecture as Section 4. The vocabulary is rebuilt from the full
MIMIC-IV sample dataset, so `word_embeddings` will be larger than the
dev-mode model.


In [ ]:
full_model = EHRMamba(
    dataset=full_sample_dataset,
    embedding_dim=128,
    num_layers=2,
    state_size=16,
    conv_kernel=4,
    dropout=0.1,
    use_ehr_mamba_embedding=True,
)
full_trainer = Trainer(model=full_model, metrics=['roc_auc', 'pr_auc'])
print(full_model)


### 19. Train


In [ ]:
full_trainer.train(
    train_dataloader=full_train_loader,
    val_dataloader=full_val_loader,
    epochs=50,
    monitor='roc_auc',
    optimizer_params={'lr': 1e-4},
)


### 20. Evaluate on Test Set


In [ ]:
full_trainer.evaluate(full_test_loader)


### 21. Get Patient Embeddings

Pooled patient representations from the full MIMIC-IV model — same API
as Section 7. Embedding shape will be `(B, 128)` regardless of dataset.


In [ ]:
full_model.eval()
with torch.no_grad():
    full_batch = next(iter(full_test_loader))
    full_output = full_model(**full_batch, embed=True)
    print(f"Patient embeddings shape: {full_output['embed'].shape}")
    print(f"Predicted probabilities:  {full_output['y_prob'].squeeze()}")
    print(f"Ground truth labels:      {full_output['y_true'].squeeze()}")
